Step 1 - Create a dataframe with manual schema

In [0]:
from pyspark.sql.types import StructType, StructField, LongType, StringType, DoubleType

# Define schema manually (good practice!)
schema = StructType([
    StructField("order_id",  LongType(),   False),
    StructField("customer",  StringType(), True),
    StructField("product",   StringType(), True),
    StructField("amount",    DoubleType(), True),
    StructField("status",    StringType(), True),
])

# Create data directly — no CSV file needed
data = [
    (1001, "Alice", "Laptop",      85000.0, "completed"),
    (1002, "Bob",   "Phone",       25000.0, "completed"),
    (1003, "Carol", "Tablet",      None,    "pending"),
    (1004, "Dave",  "Laptop",      85000.0, "completed"),
    (1005, "Eve",   "Phone",       25000.0, "cancelled"),
    (1006, "Frank", "Headphones",  5000.0,  "completed"),
]

df_raw = spark.createDataFrame(data, schema)
df_raw.show()

Step 2 - Check the column types Spark detected

In [0]:
df_raw.printSchema()

Step 3 — Transform the data

In [0]:
from pyspark.sql import functions as F

df_clean = (df_raw

    # 1. Keep only completed orders
    .filter(F.col("status") == "completed")

    # 2. Drop rows where amount is null
    .filter(F.col("amount").isNotNull())

    # 3. Select only the columns we need
    .select(
        F.col("order_id"),
        F.col("customer"),
        F.col("product"),
        F.col("amount").cast("double")  # cast to number
    )

    # 4. Add a new column: amount in USD (1 NPR = 0.0075 USD)
    .withColumn("amount_usd",
        F.round(F.col("amount") * 0.0075, 2)
    )
)

df_clean.show()

Step 4 - Write the clean DataFrame as a Delta table

In [0]:
# Write the clean DataFrame as a Delta table
# Delta = Parquet files + a transaction log (like a receipt book)

(df_clean.write
    .format("delta")
    .mode("overwrite")          # overwrite if table exists
    .saveAsTable("sales_clean")    # registers in the metastore
)

print("Delta table saved!")

# You can also save to a path instead:
# df_clean.write.format("delta").mode("overwrite")
#     .save("/delta/sales_clean")

Step 5 — Verify & query.Read back and check what was saved

In [0]:
# Option A: Query with Spark
df_check = spark.read.format("delta").table("sales_clean")
df_check.show()

# Option B: Query with SQL (works in any cell)
spark.sql("""
    SELECT product,
           COUNT(*) AS total_orders,
           ROUND(SUM(amount), 0) AS total_amount
    FROM sales_clean
    GROUP BY product
    ORDER BY total_amount DESC
""").show()

# See the Delta transaction log (history)
spark.sql("DESCRIBE HISTORY sales_clean").show()